In [ ]:
pip install langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu torch transformers pandas openpyxl pypdf -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.llms import Ollama

In [ ]:

# Configuration
DATA_PATH = "data/papers/raw/"  
FAISS_PATH = "faiss_index"
EMBED_MODEL = "all-MiniLM-L6-v2"  
LLM_MODEL = "qwen2:7b"
TOP_K = 3
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64

In [37]:
# 1. Load Data (JSON from Task 1)
def load_papers(data_dir):
    """Load papers specifically in PDF format."""
    documents = []
    
    for filename in os.listdir(data_dir):
        # Ensure only .pdf files are processed
        if filename.lower().endswith(".pdf"):
            file_path = os.path.join(data_dir, filename)
            
            try:
                # Parse PDF using LangChain's PyPDFLoader
                loader = PyPDFLoader(file_path)
                # load() splits the PDF by page, returning a list of Documents (one per page)
                pages = loader.load()
                
                # Standardize metadata to align sources for the benchmark
                for page in pages:
                    # Write the source filename uniformly into the 'source' field
                    page.metadata["source"] = filename
                    
                documents.extend(pages)
            except Exception as e:
                print(f"Error reading file {filename}: {e}")
    
    # Note: The length printed here is the number of "pages"
    print(f"Successfully loaded {len(documents)} pages of PDF content.")
    return documents

In [21]:
# 2. Text Chunking
def chunk_docs(docs):
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(docs)
    print(f"Total chunks: {len(chunks)}")
    return chunks

In [23]:
# 3. Embedding & FAISS Vector Store
def build_faiss(chunks):
    emb = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"})
    db = FAISS.from_documents(chunks, emb)
    db.save_local(FAISS_PATH)
    return db

def load_faiss():
    emb = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"})
    return FAISS.load_local(FAISS_PATH, emb, allow_dangerous_deserialization=True)

In [ ]:
# 4. Initialize Local LLM

def get_llm():
    print("Connecting to local Ollama instance...")
    # This connects to your local Ollama app running in the background
    llm = Ollama(model="qwen2:7b", temperature=0.1)
    return llm

In [25]:
# 5. Build RAG QA Chain
def build_rag_chain(retriever):
    prompt = ChatPromptTemplate.from_template("""Answer the question based only on the context:
    Context: {context}
    Question: {question}
    Answer:""")
    llm = get_llm()
    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain

In [26]:
# 6. Execute Query & Output Results
def run_qa(chain, retriever, question):
    # Get answer
    answer = chain.invoke(question)
    # Get supporting chunks
    chunks = retriever.get_relevant_documents(question)

    print(f"Question: {question}")
    print(f"\nAnswer:\n{answer}")
    print(f"\nTop-{TOP_K} Supporting Chunks:")
    for i, c in enumerate(chunks, 1):
        print(f"\n--- Chunk {i} | Source: {c.metadata['source']} ---")
        print(c.page_content)

In [46]:
import pandas as pd
import json
import os

# 1. Load Benchmark Questions
QUESTION_FILE = "evaluation/benchmarking_question.xlsx"
OUTPUT_FILE = "rag_benchmark_output.csv"

def run_benchmark_eval(retriever, rag_chain, question_file, output_file):
    print("Starting Task 2: Batch RAG generation and formatting...")
    
    # Read questions
    df_questions = pd.read_excel(question_file)
    results = []

    # 2. Iterate through questions
    for index, row in df_questions.iterrows():
        qid = row['QID']
        question = row['Question']
        target_doc_ids = row.get('Source Doc IDs', '')
        
        print(f"Processing QID: {qid} ...")
        
        # A. Retrieval
        # Get retrieved chunks (crucial for evaluating 'Retrieval failure')
        retrieved_docs = retriever.invoke(question) 
        
        retrieved_contexts = []
        retrieved_sources = []
        
        for i, doc in enumerate(retrieved_docs):
            source_name = doc.metadata.get('source', 'Unknown')
            # Format chunk context and source for LLM/human judge
            retrieved_contexts.append(f"[Chunk {i+1} | Source: {source_name}]\n{doc.page_content}")
            retrieved_sources.append(source_name)
            
        full_context = "\n\n------------------------\n\n".join(retrieved_contexts)
        
        # B. Generation
        try:
            answer = rag_chain.invoke(question) 
            # Extract text if chain returns a dict
            answer_text = answer if isinstance(answer, str) else answer.get('result', str(answer))
        except Exception as e:
            answer_text = f"Error during generation: {e}"
            
        # 3. Structure the output
        results.append({
            "QID": qid,
            "Question": question,
            "Target_Doc_IDs": target_doc_ids,  # Ground truth doc IDs
            "Generated_Answer": answer_text,   # Generated response
            "Retrieved_Context": full_context, # Context fed to LLM
            "Retrieved_Sources": json.dumps(retrieved_sources) # List of sources
        })

    # 4. Export to CSV
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\nTask 2 Complete! Output saved to: {output_file}")

# ================= Main Execution =================
if __name__ == "__main__":
    
    if os.path.exists(QUESTION_FILE):
        papers = load_papers(DATA_PATH)
        chunks = chunk_docs(papers)
        db = build_faiss(chunks)
        
        retriever = db.as_retriever(search_kwargs={"k": TOP_K})
        rag_chain = build_rag_chain(retriever)

        run_benchmark_eval(retriever, rag_chain, QUESTION_FILE, OUTPUT_FILE)
    else:
        print(f"File not found: {QUESTION_FILE}. Please check the path.")

Successfully loaded 199 pages of PDF content.
Total chunks: 1426


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20642.84it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting to local Ollama instance...
Starting Task 2: Batch RAG generation and formatting...
Processing QID: 1 ...
Processing QID: 2 ...
Processing QID: 3 ...
Processing QID: 4 ...
Processing QID: 5 ...
Processing QID: 6 ...
Processing QID: 7 ...
Processing QID: 8 ...
Processing QID: 9 ...
Processing QID: 10 ...
Processing QID: 11 ...
Processing QID: 12 ...
Processing QID: 13 ...
Processing QID: 14 ...
Processing QID: 15 ...
Processing QID: 16 ...
Processing QID: 17 ...
Processing QID: 18 ...
Processing QID: 19 ...
Processing QID: 20 ...
Processing QID: 21 ...
Processing QID: 22 ...
Processing QID: 23 ...
Processing QID: 24 ...
Processing QID: 25 ...
Processing QID: 26 ...
Processing QID: 27 ...
Processing QID: 28 ...
Processing QID: 29 ...
Processing QID: 30 ...
Processing QID: 31 ...
Processing QID: 32 ...
Processing QID: 33 ...
Processing QID: 34 ...
Processing QID: 35 ...
Processing QID: 36 ...

Task 2 Complete! Output saved to: rag_benchmark_output.csv
